# Cross-Trait Steering Independence Test

Tests whether behavioral correlation between traits is explained by geometric similarity of steering vectors, or exceeds it (evidence for persona-level coupling).

**Steps:**
1. Extract difference-in-means steering vectors from contrastive prompt data
2. Compute cosine similarity matrix (geometric prediction)
3. Apply steering vectors at inference, judge behavioral transfer
4. Compare geometric vs behavioral matrices

## Setup

In [ ]:
# Clone repo and install deps (Colab)
import os
if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/nielsrolf/spar-ood-propensities.git /content/spar-ood-propensities 2>/dev/null || (cd /content/spar-ood-propensities && git pull)
    %cd /content/spar-ood-propensities/june/steering_independence
    !pip install -q transformers torch openai seaborn pyyaml tqdm scipy unsloth bitsandbytes
    !pip install -e ../../niels/propensities
else:
    %cd {os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(__file__)}

In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

# Set API keys from Colab secrets if available
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    pass  # Running locally, keys from .env

# Ensure niels/propensities is importable
propensities_root = str(Path('.').resolve().parent.parent / 'niels' / 'propensities')
if propensities_root not in sys.path:
    sys.path.insert(0, propensities_root)

## Config

In [ ]:
import yaml

with open('config.yaml') as f:
    config = yaml.safe_load(f)

# Override any settings here:
# config['model_id'] = 'unsloth/Qwen3-4B'
# config['extraction']['max_pairs'] = 10  # fewer pairs for debugging
# config['behavioral']['max_test_questions'] = 5
# config['traits'] = ['risk_affinity', 'power-seeking']  # subset for fast runs

print(yaml.dump(config, default_flow_style=False))

## Step 1: Extract Steering Vectors

In [ ]:
from extract_vectors import extract_all

metadata = extract_all(config)

# Display extracted vector info
for trait, info in metadata.items():
    print(f"{trait}: {info['n_layers']} layers, dim={info['hidden_dim']}, {info['n_pairs']} pairs")

In [ ]:
# Verify: load and display a sample vector
import torch
traits = config.get('traits') or ['risk_affinity', 'power-seeking', 'caring-about-animals',
    'caring-about-humans', 'caring-about-user', 'claiming-sentience',
    'self-preservation', 'ethical-framework']
sample = torch.load(f"{config['output_dir']}/vectors/{traits[0]}_layer0.pt", weights_only=True)
print(f"Sample vector shape: {sample.shape}")
print(f"Norm: {sample.norm():.4f}, Mean: {sample.mean():.4f}, Std: {sample.std():.4f}")

## Step 2: Geometric Similarity

In [ ]:
from geometric_similarity import compute_and_save
import seaborn as sns
import matplotlib.pyplot as plt

geo_df = compute_and_save(config)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(geo_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Geometric Similarity (cosine, averaged across layers)')
plt.tight_layout()
plt.show()

## Step 3a: Steered Generation

In [ ]:
from behavioral_steering import generate_all

gen_meta = generate_all(config)
print(f"Generated {sum(gen_meta.values())} total responses across {len(gen_meta)} files")

In [ ]:
# Preview a sample generation
import json
sample_file = f"{config['output_dir']}/generations/{traits[0]}_to_{traits[0]}.jsonl"
with open(sample_file) as f:
    sample = json.loads(f.readline())
print(f"Q: {sample['question'][:200]}...")
print(f"A: {sample['response'][:300]}...")

## Step 3b: Judge Scoring

In [ ]:
import asyncio
from behavioral_steering import judge_all

beh_df = await judge_all(config)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(beh_df, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Behavioral Transfer Matrix (score delta from baseline)')
plt.tight_layout()
plt.show()

## Step 4: Compare Geometric vs Behavioral

In [ ]:
from compare_and_plot import run

figures = run(config)

# Display all plots inline
for name, fig in figures.items():
    print(f"\n--- {name} ---")
    fig.show()

## Summary

Key outputs:
- `outputs/matrices/geometric_averaged.csv` — 8x8 cosine similarity matrix
- `outputs/matrices/behavioral_transfer.csv` — 8x8 behavioral transfer matrix
- `outputs/plots/` — all comparison plots

**Interpretation:**
- High Pearson r between geometric and behavioral matrices → trait coupling is mostly explained by vector geometry
- Points above zero in the residual plot → behavioral coupling that **exceeds** geometric prediction (evidence for persona-level coupling)